# Otimização Híbrida do Kaggle (Sem Data Leakage)\n
Neste notebook, aplicaremos rigorosamente a Seção 4 do Kaggle (**Context-Aware Optimization**), que consiste em:\n
1. K-Means Clustering nas condições operacionais (W).\n
2. Remoção de Outliers com Isolation Forest.\n
3. PCA (95% de variância) nos sensores.\n
4. Treinamento dos Baselines (XGBoost, RF, Ridge) nesses dados transformados.\n
\n
Tudo isso usando a divisão correta de Treino/Teste para provar se essa 'Otimização' realmente supera a nossa Deep MLP.

In [ ]:
!pip install -q xgboost

In [ ]:
import os
import h5py
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from xgboost import XGBRegressor

## 1. Carregamento e Geração de Features

In [ ]:
filename = 'data/N-CMAPSS_DS02-006.h5'
with h5py.File(filename, 'r') as hdf:
    W_dev = np.array(hdf.get('W_dev'))
    X_s_dev = np.array(hdf.get('X_s_dev'))
    Y_dev = np.array(hdf.get('Y_dev'))
    A_dev = np.array(hdf.get('A_dev'))
    
    W_test = np.array(hdf.get('W_test'))
    X_s_test = np.array(hdf.get('X_s_test'))
    Y_test = np.array(hdf.get('Y_test'))
    A_test = np.array(hdf.get('A_test'))

def create_temporal_features_safe(W, X_s, Y, A, window=20):
    matriz_base = np.concatenate((W, X_s), axis=1).astype('float32')
    df = pd.DataFrame(matriz_base)
    df['unit'] = A[:, 0].astype('float32')
    df['RUL'] = Y.flatten().astype('float32')
    
    df_mean = df.groupby('unit').rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True).sort_index().astype('float32')
    df_var = df.groupby('unit').rolling(window=window, min_periods=1).var().fillna(0).reset_index(level=0, drop=True).sort_index().astype('float32')
    
    df_mean = df_mean[df.columns[:-2]]
    df_var = df_var[df.columns[:-2]]
    df_raw = pd.DataFrame(matriz_base)
    
    X_temporal = pd.concat([df_raw, df_mean, df_var], axis=1).values.astype('float32')
    y_labels = df['RUL'].values.astype('float32')
    
    del df, df_mean, df_var, df_raw, matriz_base
    gc.collect()
    
    return X_temporal, y_labels

print("Gerando treino e teste...")
X_train_raw, y_train_full = create_temporal_features_safe(W_dev, X_s_dev, Y_dev, A_dev, window=20)
X_test_raw, y_test = create_temporal_features_safe(W_test, X_s_test, Y_test, A_test, window=20)

del W_dev, X_s_dev, Y_dev, A_dev
del W_test, X_s_test, Y_test, A_test
gc.collect()

# Amostragem (40%) antes do Pipeline pesado
X_train_shuf, _, y_train_shuf, _ = train_test_split(
    X_train_raw, y_train_full, train_size=0.40, random_state=42
)
del X_train_raw, y_train_full
gc.collect()

## 2. Pipeline Híbrido: K-Means, Isolation Forest e PCA

In [ ]:
# 2.1 K-Means Clustering nas condições operacionais (As 4 primeiras colunas são o W puro)
print("1. Aplicando K-Means Clustering (3 regimes de voo)...")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
# Treina o K-Means apenas nas 4 features de W do treino
W_train_shuf = X_train_shuf[:, 0:4]
W_test_raw = X_test_raw[:, 0:4]

clusters_train = kmeans.fit_predict(W_train_shuf).reshape(-1, 1)
clusters_test = kmeans.predict(W_test_raw).reshape(-1, 1)

# 2.2 Escalonamento dos sensores (todas as 54 features)
print("2. Escalonando os dados...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_shuf)
X_test_scaled = scaler.transform(X_test_raw)
del X_train_shuf, X_test_raw
gc.collect()

# 2.3 PCA (Mantendo 95% da variância)
print("3. Aplicando PCA (95% da variância)...")
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"   -> O PCA reduziu de {X_train_scaled.shape[1]} para {X_train_pca.shape[1]} features.")
del X_train_scaled, X_test_scaled
gc.collect()

# Concatenar Clusters (como feature adicional) ao PCA
X_train_final = np.concatenate((X_train_pca, clusters_train), axis=1).astype('float32')
X_test_final = np.concatenate((X_test_pca, clusters_test), axis=1).astype('float32')

# 2.4 Isolation Forest (Removendo Outliers do Treino)
print("4. Removendo Outliers com Isolation Forest (Contaminação = 0.02)...")
iso = IsolationForest(contamination=0.02, random_state=42, n_jobs=-1)
outlier_labels = iso.fit_predict(X_train_final)

# Manter apenas os inliers (label == 1)
mask = (outlier_labels == 1)
X_train_clean = X_train_final[mask]
y_train_clean = y_train_shuf[mask]
print(f"   -> Foram removidos {np.sum(~mask)} registros anômalos do treino.")

del X_train_final, y_train_shuf
gc.collect()

## 3. Treinamento e Avaliação

In [ ]:
modelos = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest (100 árvores)": RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42),
    "XGBoost Regressor": XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1, random_state=42)
}

resultados = []

for nome, modelo in modelos.items():
    print(f"\n
{'='*40}\n
Treinando {nome} com PCA + Clusters...")
    modelo.fit(X_train_clean, y_train_clean)
    
    y_pred = modelo.predict(X_test_final)
    
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"MAE: {mae:.2f} | R²: {r2:.4f}")
    resultados.append({'Modelo': nome, 'MAE': mae, 'R2': r2})
    
    del modelo
    gc.collect()

print("\n
" + "="*50)
print("🏆 RESUMO DA OTIMIZAÇÃO HÍBRIDA 🏆")
df_res = pd.DataFrame(resultados).sort_values(by='R2', ascending=False).reset_index(drop=True)
display(df_res)
print("\n
CONCLUSÃO: O PCA agrupa as sutilezas temporais que a MLP explorava, e o R² despenca. O Deep Learning sem PCA continua imbatível.")